# Lecture 14 — Variational Autoencoders (VAE)

**PHYG004 / PHY5006, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

## Learning goals

| # | Goal |
|---|------|
| 1 | Build a plain **autoencoder** and see why it can't generate new data |
| 2 | Understand the **one key idea** that turns an AE into a VAE |
| 3 | Implement a **VAE** from scratch in JAX and train it on MNIST |
| 4 | Explore the **latent space** — scatter, grid, interpolation |
| 5 | Experiment with the **reconstruction vs regularisation** trade-off (β-VAE) |
| 6 | Read the loss as a **variational free energy**: $-\text{ELBO} = E_q + \text{KL}$ |
| 7 | **Physics demo**: train a VAE on 2D Ising configs and watch a latent axis discover **magnetisation** |

> **Runtime.** Standard Colab CPU is enough for the MNIST part (1–3 min) and for
> the Ising demo (the Metropolis sampler runs in a few CPU-seconds). A T4 GPU
> only makes it faster — no GPU is required.

> **Physicist's view.** Keep one analogy in mind throughout: the VAE loss *is* a
> **free energy**. Reconstruction error plays the role of an energy (how well the
> model fits the data), the KL term plays the role of an entropic / prior
> constraint, and minimising the loss is minimising $F = \langle E\rangle - TS$
> with $\beta$ setting the "temperature" of the trade-off.


## 0. Setup

In [ ]:
# On Google Colab, uncomment the next line to install dependencies.
# !pip install -q jax jaxlib optax flax matplotlib

import jax
import jax.numpy as jnp
import jax.random as jr
import jax.lax as lax
import optax
import flax.nnx as nnx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# Default PRNG key for the whole notebook (CLAUDE.md convention).
KEY = jr.PRNGKey(42)

print(f"JAX backend: {jax.default_backend()}")
print(f"JAX version: {jax.__version__}")


## 1. What is a generative model?

So far we've built **discriminative** models: given input $x$, predict output $y$.

A **generative** model learns the data distribution itself — it can **create new samples** that look like the training data.

| Discriminative | Generative |
|---|---|
| "This image is a **7**" | "Here's a **new** image of a 7" |
| Input → Label | Random noise → New data |
| $p(y \mid x)$ | $p(x)$ |

Why does this matter? Generative models power image synthesis (DALL·E, Stable Diffusion), protein design, new materials discovery, and more.


In [ ]:
# Visual: discriminative vs generative
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

for ax in axes:
    ax.set_xlim(0, 10); ax.set_ylim(0, 4); ax.axis("off")

def box(ax, x, y, w, h, text, fc, fontsize=11):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                fc=fc, ec="#37474f", lw=1.4))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize,
            wrap=True)

def arrow(ax, x1, y1, x2, y2, color="#37474f"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.8))

# Left: discriminative
ax = axes[0]
ax.set_title("Discriminative model", fontsize=13, fontweight="bold", pad=12)
box(ax, 0.3, 1.3, 2.2, 1.4, "Image\n(input)", "#e3f2fd")
box(ax, 4.0, 1.3, 2.2, 1.4, "Neural\nNetwork", "#fff3e0")
box(ax, 7.5, 1.3, 2.2, 1.4, 'Label\n"7"', "#e8f5e9")
arrow(ax, 2.5, 2.0, 4.0, 2.0)
arrow(ax, 6.2, 2.0, 7.5, 2.0)

# Right: generative
ax = axes[1]
ax.set_title("Generative model", fontsize=13, fontweight="bold", pad=12)
box(ax, 0.3, 1.3, 2.2, 1.4, "Random\nnoise", "#fce4ec")
box(ax, 4.0, 1.3, 2.2, 1.4, "Neural\nNetwork", "#fff3e0")
box(ax, 7.5, 1.3, 2.2, 1.4, "New image!\n(output)", "#e8f5e9")
arrow(ax, 2.5, 2.0, 4.0, 2.0)
arrow(ax, 6.2, 2.0, 7.5, 2.0)

plt.tight_layout(); plt.show()


## 2. Load and prepare MNIST

We'll work with **MNIST digits** — 28×28 grayscale images. We binarise them (pixel = 0 or 1) to keep things simple.


In [ ]:
def load_mnist():
    try:
        from tensorflow.keras.datasets import mnist as km
        return km.load_data()
    except Exception:
        pass
    try:
        from keras.datasets import mnist as km
        return km.load_data()
    except Exception:
        pass
    import urllib.request, os, tempfile
    url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
    path = os.path.join(tempfile.gettempdir(), "mnist.npz")
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    with np.load(path, allow_pickle=True) as f:
        return (f["x_train"], f["y_train"]), (f["x_test"], f["y_test"])

(x_train_raw, y_train), (x_test_raw, y_test) = load_mnist()

def binarise(x):
    return (x.astype("float32") / 255.0 > 0.5).astype("float32").reshape(-1, 784)

x_train = binarise(x_train_raw)
x_test  = binarise(x_test_raw)
print(f"Training: {x_train.shape}  Test: {x_test.shape}")

fig, axes = plt.subplots(1, 10, figsize=(12, 1.4))
for ax, img, lab in zip(axes, x_train[:10], y_train[:10]):
    ax.imshow(img.reshape(28, 28), cmap="gray_r")
    ax.set_title(int(lab), fontsize=11); ax.axis("off")
plt.suptitle("Binarised MNIST samples", fontsize=13, y=1.05)
plt.tight_layout(); plt.show()


## 3. Warm-up: the plain Autoencoder (AE)

Before building a VAE, let's build a regular **autoencoder**. It compresses data into a small "bottleneck" (latent code) and tries to reconstruct the original.

**Key idea:** force the network through a narrow 2D bottleneck. If reconstruction works, the network has learned a useful compression.


In [ ]:
# Diagram: autoencoder architecture
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.set_xlim(0, 14); ax.set_ylim(0, 4); ax.axis("off")
ax.set_title("Autoencoder: compress → bottleneck → reconstruct", fontsize=13,
             fontweight="bold", pad=10)

# Encoder side
widths  = [1.8, 1.3, 0.6]
heights = [2.4, 1.8, 0.8]
labels  = ["Input\n784 pixels", "Hidden\n256", "Latent\n2D"]
colors  = ["#e3f2fd", "#bbdefb", "#90caf9"]
xs = [0.5, 3.0, 5.2]

for x, w, h, lab, fc in zip(xs, widths, heights, labels, colors):
    y = 2.0 - h/2
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                fc=fc, ec="#1565c0", lw=1.4))
    ax.text(x + w/2, 2.0, lab, ha="center", va="center", fontsize=10)

# Decoder side (mirror)
widths_d  = [0.6, 1.3, 1.8]
heights_d = [0.8, 1.8, 2.4]
labels_d  = ["", "Hidden\n256", "Output\n784 pixels"]
colors_d  = ["#90caf9", "#c8e6c9", "#a5d6a7"]
xs_d = [7.2, 8.8, 11.2]

for x, w, h, lab, fc in zip(xs_d, widths_d, heights_d, labels_d, colors_d):
    y = 2.0 - h/2
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                fc=fc, ec="#2e7d32", lw=1.4))
    if lab:
        ax.text(x + w/2, 2.0, lab, ha="center", va="center", fontsize=10)

# Arrows
for x1, x2 in [(2.3, 3.0), (4.3, 5.2), (5.8, 7.2), (7.8, 8.8), (10.1, 11.2)]:
    arrow(ax, x1, 2.0, x2, 2.0)

# Labels
ax.text(3.0, 3.6, "Encoder", ha="center", fontsize=12, color="#1565c0",
        fontweight="bold")
ax.text(10.5, 3.6, "Decoder", ha="center", fontsize=12, color="#2e7d32",
        fontweight="bold")
ax.text(6.5, 3.2, "Bottleneck", ha="center", fontsize=11, color="#c62828",
        fontweight="bold")

plt.tight_layout(); plt.show()


### 3.1 Build the autoencoder

In [ ]:
class AE_Encoder(nnx.Module):
    def __init__(self, in_dim=784, hidden=256, latent=2, *, rngs):
        self.fc1 = nnx.Linear(in_dim, hidden, rngs=rngs)
        self.fc2 = nnx.Linear(hidden, latent, rngs=rngs)

    def __call__(self, x):
        h = nnx.relu(self.fc1(x))
        return self.fc2(h)   # latent code z (deterministic)


class AE_Decoder(nnx.Module):
    def __init__(self, latent=2, hidden=256, out_dim=784, *, rngs):
        self.fc1 = nnx.Linear(latent, hidden, rngs=rngs)
        self.fc2 = nnx.Linear(hidden, out_dim, rngs=rngs)

    def __call__(self, z):
        h = nnx.relu(self.fc1(z))
        return self.fc2(h)  # logits


class Autoencoder(nnx.Module):
    def __init__(self, in_dim=784, hidden=256, latent=2, *, rngs):
        self.encoder = AE_Encoder(in_dim, hidden, latent, rngs=rngs)
        self.decoder = AE_Decoder(latent, hidden, in_dim, rngs=rngs)

    def __call__(self, x):
        z = self.encoder(x)
        logits = self.decoder(z)
        return logits, z


ae = Autoencoder(rngs=nnx.Rngs(0))
n_params = sum(p.size for p in jax.tree.leaves(nnx.state(ae, nnx.Param)))
print(f"Autoencoder parameters: {n_params:,}")


### 3.2 Train the autoencoder

In [ ]:
def train_ae(model, x_train, n_epochs=10, batch_size=128, lr=1e-3, seed=0):
    optimizer = nnx.Optimizer(model, optax.adam(lr), wrt=nnx.Param)

    @nnx.jit
    def train_step(model, optimizer, x):
        def loss_fn(model, x):
            logits, z = model(x)
            bce = optax.sigmoid_binary_cross_entropy(logits, x).sum(axis=-1)
            return bce.mean()
        loss, grads = nnx.value_and_grad(loss_fn, argnums=nnx.DiffState(0, nnx.Param))(
            model, x)
        optimizer.update(model, grads)
        return loss

    n = len(x_train)
    history = []
    key = jr.PRNGKey(seed)

    for epoch in range(n_epochs):
        key, sub = jr.split(key)
        perm = jr.permutation(sub, n)
        x_shuf = x_train[perm]
        ep_loss = 0.0; n_batches = 0
        for i in range(0, n, batch_size):
            xb = jnp.asarray(x_shuf[i:i+batch_size])
            loss = train_step(model, optimizer, xb)
            ep_loss += float(loss); n_batches += 1
        ep_loss /= n_batches
        history.append(ep_loss)
        print(f"epoch {epoch+1:2d}  loss = {ep_loss:.2f}")
    return history

ae = Autoencoder(rngs=nnx.Rngs(0))
ae_hist = train_ae(ae, x_train, n_epochs=10)


### 3.3 See the reconstructions

In [ ]:
# Reconstruct some test images
@nnx.jit
def ae_reconstruct(model, x):
    logits, z = model(x)
    return jax.nn.sigmoid(logits)

x_sample = jnp.asarray(x_test[:8])
x_recon = np.asarray(ae_reconstruct(ae, x_sample))

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(x_sample[i].reshape(28, 28), cmap="gray_r"); axes[0, i].axis("off")
    axes[1, i].imshow(x_recon[i].reshape(28, 28), cmap="gray_r"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Reconstructed", fontsize=11)
plt.suptitle("Autoencoder: original vs reconstructed", fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


### 3.4 Look at the latent space

In [ ]:
# Encode the test set and visualise the 2D latent space
@nnx.jit
def ae_encode(model, x):
    return model.encoder(x)

z_test_ae = np.asarray(ae_encode(ae, jnp.asarray(x_test)))

plt.figure(figsize=(6.5, 5.5))
sc = plt.scatter(z_test_ae[:, 0], z_test_ae[:, 1], c=y_test, cmap="tab10",
                 s=4, alpha=0.5)
plt.colorbar(sc, label="digit", ticks=range(10))
plt.xlabel("$z_1$"); plt.ylabel("$z_2$")
plt.title("Autoencoder latent space (test set)", fontsize=13)
plt.tight_layout(); plt.show()


## 4. The problem: autoencoders can't generate

The autoencoder compresses well, but look what happens when we try to **generate** new digits by picking random points in the latent space:


In [ ]:
# Try to generate from a random autoencoder latent
@nnx.jit
def ae_decode(model, z):
    return jax.nn.sigmoid(model.decoder(z))

# Sample random z values — but where? We need to know the range.
z_min = z_test_ae.min(axis=0)
z_max = z_test_ae.max(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: latent space with "holes"
ax = axes[0]
ax.scatter(z_test_ae[:, 0], z_test_ae[:, 1], c=y_test, cmap="tab10", s=3, alpha=0.4)
ax.set_title("AE latent space has gaps and clusters", fontsize=12)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")

# Draw some random points and decode them
key = jr.PRNGKey(42)
z_random = jr.uniform(key, (16, 2), minval=z_min - 1, maxval=z_max + 1)
ax.scatter(z_random[:, 0], z_random[:, 1], c="red", s=60, marker="x", linewidths=2,
           label="random sample points")
ax.legend(fontsize=10)

# Right: decoded random points — messy
imgs = np.asarray(ae_decode(ae, z_random))
ax = axes[1]
n_show = 16
grid_h, grid_w = 4, 4
canvas = np.zeros((grid_h * 28, grid_w * 28))
for i in range(n_show):
    r, c = divmod(i, grid_w)
    canvas[r*28:(r+1)*28, c*28:(c+1)*28] = imgs[i].reshape(28, 28)
ax.imshow(canvas, cmap="gray_r")
ax.set_title("Decoded random points → mostly garbage", fontsize=12)
ax.axis("off")

plt.tight_layout(); plt.show()


**The problem is clear:**

1. The latent space has **gaps** — regions with no training data
2. Points in the gaps decode to **meaningless blobs**
3. The latent codes have **no predictable structure** — we don't know where to sample

We need a way to make the latent space **smooth and well-organised**, so that *any* random point decodes to a reasonable image.


## 5. The key idea: from Autoencoder to VAE

The **Variational Autoencoder** (Kingma & Welling, 2014) makes two changes:


In [ ]:
# Diagram: AE vs VAE side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax in axes:
    ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis("off")

# === LEFT: regular AE ===
ax = axes[0]
ax.set_title("Regular Autoencoder", fontsize=14, fontweight="bold", pad=10)

box(ax, 0.3, 1.8, 2.0, 1.4, "Input $x$", "#e3f2fd")
box(ax, 3.3, 1.8, 2.2, 1.4, "Encoder", "#bbdefb")
box(ax, 6.5, 1.8, 2.0, 1.4, "Latent $z$\n(a point)", "#fff9c4", fontsize=10)
box(ax, 9.5, 1.8, 2.2, 1.4, "Decoder", "#c8e6c9")
arrow(ax, 2.3, 2.5, 3.3, 2.5)
arrow(ax, 5.5, 2.5, 6.5, 2.5)
arrow(ax, 8.5, 2.5, 9.5, 2.5)

ax.text(6.0, 0.6, "Loss = reconstruction only", fontsize=11, ha="center",
        color="#c62828", fontweight="bold")

# === RIGHT: VAE ===
ax = axes[1]
ax.set_title("Variational Autoencoder (VAE)", fontsize=14, fontweight="bold", pad=10)

box(ax, 0.3, 1.8, 2.0, 1.4, "Input $x$", "#e3f2fd")
box(ax, 3.3, 1.8, 2.2, 1.4, "Encoder", "#bbdefb")

# mu and sigma outputs
box(ax, 6.2, 3.2, 1.5, 0.9, "$\\mu, \\sigma$", "#e1bee7", fontsize=10)
box(ax, 6.2, 0.9, 1.5, 0.9, "sample $z$", "#fce4ec", fontsize=10)
box(ax, 8.8, 1.8, 2.2, 1.4, "Decoder", "#c8e6c9")

arrow(ax, 2.3, 2.5, 3.3, 2.5)
arrow(ax, 5.5, 2.7, 6.2, 3.5)
arrow(ax, 6.95, 3.2, 6.95, 1.8)
arrow(ax, 7.7, 1.35, 8.8, 2.2)

ax.text(6.0, 0.15, "Loss = reconstruction + KL regularisation",
        fontsize=11, ha="center", color="#c62828", fontweight="bold")
ax.text(8.0, 3.7, "distribution,\nnot a point!", fontsize=9, color="#7b1fa2",
        fontstyle="italic", ha="center")

plt.tight_layout(); plt.show()


### The two changes

**Change 1: The encoder outputs a *distribution*, not a point.**

Instead of mapping $x$ to a single latent code $z$, the encoder outputs the **mean** $\mu$ and **standard deviation** $\sigma$ of a Gaussian distribution. We then *sample* $z$ from this distribution:

$$z \sim \mathcal{N}(\mu, \sigma^2)$$

This forces nearby inputs to have overlapping latent distributions → **smooth latent space**.

**Change 2: Add a regularisation term to the loss.**

$$\text{VAE Loss} = \underbrace{\text{Reconstruction error}}_{\text{make output look like input}} + \underbrace{\text{KL divergence}}_{\text{keep latent space well-organised}}$$

The KL term pushes the latent distributions towards a standard Gaussian $\mathcal{N}(0, 1)$, preventing the encoder from spreading codes far apart with big gaps.

That's it! These two changes are the entire difference between an AE and a VAE.


## 6. The reparameterization trick

There's one technical detail: how do we backpropagate through a random sample?

We can't take the gradient of "draw a random number." The trick is to rewrite the sampling as:

$$z = \mu + \sigma \times \epsilon, \qquad \epsilon \sim \mathcal{N}(0, 1)$$

Now the randomness ($\epsilon$) is separated from the learnable parameters ($\mu$, $\sigma$), and gradients flow through just fine.


In [ ]:
# Diagram: reparameterization trick
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

for ax in axes:
    ax.set_xlim(0, 10); ax.set_ylim(0, 4.5); ax.axis("off")

# LEFT: naive (can't backprop)
ax = axes[0]
ax.set_title("Naive: can't backpropagate", fontsize=12, fontweight="bold",
             color="#c62828", pad=10)
box(ax, 0.5, 1.6, 2.0, 1.3, "Encoder\n→ $\\mu, \\sigma$", "#e3f2fd")
box(ax, 4.0, 1.6, 2.0, 1.3, "Sample\n$z \\sim \\mathcal{N}(\\mu, \\sigma^2)$",
    "#fce4ec", fontsize=9)
box(ax, 7.5, 1.6, 2.0, 1.3, "Decoder\n→ $\\hat{x}$", "#c8e6c9")
arrow(ax, 2.5, 2.25, 4.0, 2.25)
arrow(ax, 6.0, 2.25, 7.5, 2.25)
ax.plot([3.25, 3.25], [1.0, 1.5], color="#c62828", lw=3)
ax.plot([3.0, 3.5], [1.2, 1.2], color="#c62828", lw=3)
ax.plot([3.0, 3.5], [1.0, 1.5], color="#c62828", lw=2)
ax.plot([3.0, 3.5], [1.5, 1.0], color="#c62828", lw=2)
ax.text(5.0, 0.5, "Gradient blocked by random sampling!", fontsize=10,
        color="#c62828", ha="center")

# RIGHT: reparameterized
ax = axes[1]
ax.set_title("Reparameterized: gradient flows!", fontsize=12, fontweight="bold",
             color="#2e7d32", pad=10)
box(ax, 0.3, 1.6, 2.0, 1.3, "Encoder\n→ $\\mu, \\sigma$", "#e3f2fd")

# epsilon from outside
box(ax, 3.2, 3.2, 1.8, 0.9, "$\\epsilon \\sim \\mathcal{N}(0,1)$\n(fixed noise)",
    "#e0f2f1", fontsize=9)

box(ax, 3.5, 1.6, 2.5, 1.3, "$z = \\mu + \\sigma \\cdot \\epsilon$",
    "#f3e5f5", fontsize=10)
box(ax, 7.2, 1.6, 2.0, 1.3, "Decoder\n→ $\\hat{x}$", "#c8e6c9")

arrow(ax, 2.3, 2.25, 3.5, 2.25)
arrow(ax, 4.1, 3.2, 4.5, 2.9)
arrow(ax, 6.0, 2.25, 7.2, 2.25)

ax.annotate("", xy=(7.0, 0.9), xytext=(0.5, 0.9),
            arrowprops=dict(arrowstyle="-|>", color="#2e7d32", lw=2.0,
                            linestyle="--"))
ax.text(3.75, 0.45, "Gradient flows through $\\mu$ and $\\sigma$!",
        fontsize=10, color="#2e7d32", ha="center")

plt.tight_layout(); plt.show()


## 7. The VAE loss function

### 7.0 Where does this loss come from? The ELBO

We never wrote down the loss by hand — it falls out of one goal: **maximise the
likelihood** $p_\theta(x)$ of the data. The catch is that

$$p_\theta(x) = \int p_\theta(x \mid z)\, p(z)\, dz$$

is an intractable integral over all latent codes $z$. The variational trick is to
introduce the encoder $q_\phi(z\mid x)$ as an *approximate posterior* and apply
**Jensen's inequality** ($\log$ is concave, so $\log E[\cdot] \ge E[\log \cdot]$):

$$
\log p_\theta(x)
= \log E_{q_\phi(z\mid x)}\!\left[\frac{p_\theta(x\mid z)\, p(z)}{q_\phi(z\mid x)}\right]
\;\ge\;
\underbrace{E_{q_\phi(z\mid x)}\big[\log p_\theta(x\mid z)\big]
- \mathrm{KL}\!\big(q_\phi(z\mid x)\,\|\,p(z)\big)}_{\textstyle \text{ELBO}(x)}.
$$

The right-hand side is the **Evidence Lower BOund (ELBO)**. Maximising the ELBO
pushes the (intractable) log-likelihood up. Flipping the sign gives the loss we
actually minimise:

$$
\boxed{\;
-\,\text{ELBO}(x)
= \underbrace{-\,E_{q_\phi}\big[\log p_\theta(x\mid z)\big]}_{\text{reconstruction error}}
\;+\;
\underbrace{\mathrm{KL}\!\big(q_\phi(z\mid x)\,\|\,p(z)\big)}_{\text{KL regulariser}}
\;=\;\text{variational free energy.}
\;}
$$

**Physicist's reading.** This is exactly a **free energy** $F = \langle E\rangle - TS$:
the reconstruction term is the expected "energy" $\langle E\rangle = -E_q[\log p_\theta(x\mid z)]$,
and the KL term is the entropic / prior penalty that keeps the posterior from
collapsing onto the data. Training a VAE = minimising a variational free energy.
The two terms below are not an arbitrary recipe — they are precisely the two
pieces of $-\text{ELBO}$.

### 7.1 The two terms, concretely

For a Bernoulli decoder and a Gaussian encoder/prior, the loss has exactly two terms:

$$\text{Loss} = \underbrace{\text{BCE}(x, \hat{x})}_{\text{Reconstruction}} + \underbrace{\frac{1}{2}\sum_i \left(\mu_i^2 + \sigma_i^2 - \log \sigma_i^2 - 1\right)}_{\text{KL divergence}}$$

| Term | What it does | Analogy |
|------|-------------|---------|
| **Reconstruction** (BCE) | Make output look like input | "Memorise the data" |
| **KL divergence** | Keep $(\mu, \sigma)$ close to $\mathcal{N}(0, 1)$ | "Stay organised" |

The two terms **compete**: reconstruction wants to use all of latent space freely; KL wants to keep everything near the origin. The balance between them is what makes the latent space smooth and useful.


In [ ]:
# Visualise what KL does: penalises distributions far from N(0,1)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

x_vals = np.linspace(-5, 5, 300)

# Labels use the exact closed-form KL(N(mu, sigma^2) || N(0,1))
#   = 0.5 * (mu^2 + sigma^2 - ln sigma^2 - 1).
# N(2, 0.5^2):  0.5*(4 + 0.25 - ln 0.25 - 1) = 2.3181  -> 2.32
# N(-1, 2^2):   0.5*(1 + 4    - ln 4    - 1) = 1.3069  -> 1.31
cases = [
    (0.0, 1.0, "KL = 0.00\n(perfect match)"),
    (2.0, 0.5, "KL = 2.32\n(shifted & narrow)"),
    (-1.0, 2.0, "KL = 1.31\n(shifted & wide)"),
]

def kl_gauss(mu, sigma):
    return 0.5 * (mu**2 + sigma**2 - np.log(sigma**2) - 1.0)
# Sanity check that the labels match the formula (printed, not hard-coded):
for mu, sigma, _ in cases:
    print(f"KL(N({mu:>4}, {sigma}^2) || N(0,1)) = {kl_gauss(mu, sigma):.4f}")

for ax, (mu, sigma, title) in zip(axes, cases):
    # Prior N(0,1)
    prior = np.exp(-0.5 * x_vals**2) / np.sqrt(2 * np.pi)
    # Posterior N(mu, sigma^2)
    posterior = np.exp(-0.5 * ((x_vals - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

    ax.fill_between(x_vals, prior, alpha=0.3, color="#1565c0", label="Prior $\\mathcal{N}(0,1)$")
    ax.fill_between(x_vals, posterior, alpha=0.3, color="#c62828",
                    label=f"Encoder $\\mathcal{{N}}({mu},{sigma**2:.1f})$")
    ax.set_title(title, fontsize=11)
    ax.set_xlim(-5, 5); ax.set_ylim(0, 1.0)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.2)

plt.suptitle("KL divergence measures how far the encoder distribution is from the prior",
             fontsize=12, y=1.03)
plt.tight_layout(); plt.show()


## 8. Build the VAE

Now let's implement it. The code is very similar to the autoencoder — the main differences are:
1. Encoder outputs **two** vectors: $\mu$ and $\log\sigma$
2. We **sample** $z$ using the reparameterization trick
3. Loss includes the **KL term**


In [ ]:
class Encoder(nnx.Module):
    def __init__(self, in_dim=784, hidden=256, latent=2, *, rngs):
        self.fc1 = nnx.Linear(in_dim, hidden, rngs=rngs)
        self.fc2 = nnx.Linear(hidden, hidden, rngs=rngs)
        self.mu = nnx.Linear(hidden, latent, rngs=rngs)         # mean
        self.log_sigma = nnx.Linear(hidden, latent, rngs=rngs)  # log std

    def __call__(self, x):
        h = nnx.relu(self.fc1(x))
        h = nnx.relu(self.fc2(h))
        return self.mu(h), self.log_sigma(h)


class Decoder(nnx.Module):
    def __init__(self, latent=2, hidden=256, out_dim=784, *, rngs):
        self.fc1 = nnx.Linear(latent, hidden, rngs=rngs)
        self.fc2 = nnx.Linear(hidden, hidden, rngs=rngs)
        self.fc3 = nnx.Linear(hidden, out_dim, rngs=rngs)

    def __call__(self, z):
        h = nnx.relu(self.fc1(z))
        h = nnx.relu(self.fc2(h))
        return self.fc3(h)  # Bernoulli logits


class VAE(nnx.Module):
    def __init__(self, in_dim=784, hidden=256, latent=2, *, rngs):
        self.encoder = Encoder(in_dim, hidden, latent, rngs=rngs)
        self.decoder = Decoder(latent, hidden, in_dim, rngs=rngs)
        self.latent = latent

    def __call__(self, x, key):
        mu, log_sigma = self.encoder(x)
        # Reparameterization trick: z = mu + sigma * epsilon
        eps = jr.normal(key, mu.shape)
        z = mu + jnp.exp(log_sigma) * eps
        logits = self.decoder(z)
        return logits, mu, log_sigma


vae = VAE(rngs=nnx.Rngs(0))
n_params = sum(p.size for p in jax.tree.leaves(nnx.state(vae, nnx.Param)))
print(f"VAE parameters: {n_params:,}")


### 8.1 The loss function

In [ ]:
def vae_loss(model: VAE, x, key, beta: float = 1.0):
    logits, mu, log_sigma = model(x, key)

    # Term 1: Reconstruction (binary cross-entropy, summed over 784 pixels)
    recon = optax.sigmoid_binary_cross_entropy(logits, x).sum(axis=-1)  # (B,)

    # Term 2: KL divergence (closed-form for Gaussian → Gaussian)
    kl = 0.5 * jnp.sum(mu**2 + jnp.exp(2 * log_sigma) - 2 * log_sigma - 1.0, axis=-1)

    loss = recon + beta * kl
    return loss.mean(), {"recon": recon.mean(), "kl": kl.mean()}


### 8.2 Checkpoint exercises (TODO-1, TODO-2)

The cells above already contain a *complete, working* implementation so the rest
of the notebook runs end-to-end. Before moving on, make sure you can **derive and
re-implement the two pieces yourself** — these are the heart of the VAE.

> **TODO-1 — the KL term (second term of the ELBO).**
> In `vae_loss`, the regulariser is the closed-form Gaussian-to-Gaussian KL
> $$\mathrm{KL}\big(\mathcal{N}(\mu,\sigma^2)\,\|\,\mathcal{N}(0,1)\big)
> = \tfrac12\sum_i\big(\mu_i^2 + \sigma_i^2 - \log\sigma_i^2 - 1\big).$$
> *Hint:* the code stores `log_sigma`, so $\sigma_i^2 = \exp(2\,\log\sigma_i)$ and
> $\log\sigma_i^2 = 2\,\log\sigma_i$. Fill the blank in `kl_term_exercise` below and
> check it matches the reference.
>
> **TODO-2 — the reparameterization trick (inside the encoder/forward pass).**
> Sampling $z\sim\mathcal{N}(\mu,\sigma^2)$ must be written as
> $z = \mu + \sigma\,\epsilon,\ \epsilon\sim\mathcal{N}(0,I)$ so gradients flow
> through $\mu,\sigma$. Fill the blank in `reparameterize` below.
>
> **Checkpoint (physics-based):** with both blanks filled, a freshly trained VAE
> must reach `KL > 0` (the prior is actually used) and a total loss close to the
> reference run in §9. If `KL ≈ 0` for many epochs, your posterior collapsed —
> re-check the sign of the KL term.


In [ ]:
# ---- TODO-1: implement the Gaussian KL, then compare to the reference ----
def kl_term_exercise(mu, log_sigma):
    # TODO-1: replace `...` with the closed-form KL summed over latent dims.
    #   KL = 0.5 * sum_i( mu_i^2 + sigma_i^2 - log sigma_i^2 - 1 )
    #   Remember sigma_i^2 = exp(2 * log_sigma_i) and log sigma_i^2 = 2 * log_sigma_i.
    # kl = ...                                          # <-- fill me in
    kl = 0.5 * jnp.sum(mu**2 + jnp.exp(2 * log_sigma) - 2 * log_sigma - 1.0, axis=-1)
    return kl

def kl_reference(mu, log_sigma):
    return 0.5 * jnp.sum(mu**2 + jnp.exp(2 * log_sigma) - 2 * log_sigma - 1.0, axis=-1)

_mu  = jr.normal(jr.PRNGKey(1), (5, 2))
_ls  = 0.3 * jr.normal(jr.PRNGKey(2), (5, 2))
ok_kl = jnp.allclose(kl_term_exercise(_mu, _ls), kl_reference(_mu, _ls), atol=1e-5)
print("TODO-1 KL matches reference:", bool(ok_kl))
assert ok_kl, "TODO-1: your KL does not match the closed form yet."


In [ ]:
# ---- TODO-2: implement the reparameterization trick ----
def reparameterize(mu, log_sigma, key):
    eps = jr.normal(key, mu.shape)            # epsilon ~ N(0, I)
    # TODO-2: combine mu, sigma and eps into a sample z.
    #   z = mu + sigma * eps,  with sigma = exp(log_sigma)
    # z = ...                                 # <-- fill me in
    z = mu + jnp.exp(log_sigma) * eps
    return z

# Quick statistical check: with mu=0, log_sigma=0 the samples should look like N(0,1).
_z = reparameterize(jnp.zeros((10000, 2)), jnp.zeros((10000, 2)), jr.PRNGKey(0))
print(f"sample mean ~ 0 :  {float(_z.mean()): .3f}")
print(f"sample std  ~ 1 :  {float(_z.std()): .3f}")
assert abs(float(_z.mean())) < 0.05 and abs(float(_z.std()) - 1.0) < 0.05, \
    "TODO-2: reparameterized samples are not N(0,1)."
print("TODO-2 reparameterization trick: OK")


## 9. Train the VAE

In [ ]:
def train_vae(model, x_train, n_epochs=10, batch_size=128, lr=1e-3, beta=1.0, seed=0):
    optimizer = nnx.Optimizer(model, optax.adam(lr), wrt=nnx.Param)

    @nnx.jit
    def train_step(model, optimizer, x, key, beta):
        (loss, aux), grads = nnx.value_and_grad(vae_loss, has_aux=True,
                                                argnums=nnx.DiffState(0, nnx.Param))(
            model, x, key, beta)
        optimizer.update(model, grads)
        return loss, aux

    n = len(x_train)
    history = {"loss": [], "recon": [], "kl": []}
    key = jr.PRNGKey(seed)

    for epoch in range(n_epochs):
        key, sub = jr.split(key)
        perm = jr.permutation(sub, n)
        x_shuf = x_train[perm]

        ep = {"loss": 0.0, "recon": 0.0, "kl": 0.0}
        n_batches = 0
        for i in range(0, n, batch_size):
            xb = jnp.asarray(x_shuf[i:i+batch_size])
            key, sub = jr.split(key)
            loss, aux = train_step(model, optimizer, xb, sub, beta)
            ep["loss"]  += float(loss)
            ep["recon"] += float(aux["recon"])
            ep["kl"]    += float(aux["kl"])
            n_batches += 1

        for k in ep: ep[k] /= n_batches; history[k].append(ep[k])
        print(f"epoch {epoch+1:2d}  loss = {ep['loss']:7.2f}   "
              f"recon = {ep['recon']:7.2f}   KL = {ep['kl']:6.3f}")

    return history

vae = VAE(latent=2, rngs=nnx.Rngs(0))
hist = train_vae(vae, x_train, n_epochs=15, batch_size=128, lr=1e-3, beta=1.0)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
titles = ["Total loss", "Reconstruction (BCE)", "KL divergence"]
keys = ["loss", "recon", "kl"]
colors = ["#37474f", "#c62828", "#1565c0"]

for ax, title, k, c in zip(axes, titles, keys, colors):
    ax.plot(hist[k], "-o", ms=4, color=c)
    ax.set_xlabel("epoch"); ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 10. Explore the VAE latent space

Now the exciting part. Let's see what the VAE has learned.


### 10.1 Encode the test set — where do digits live?

In [ ]:
@nnx.jit
def encode_mean(model, x):
    mu, _ = model.encoder(x)
    return mu

mu_test = np.asarray(encode_mean(vae, jnp.asarray(x_test)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# LEFT: AE latent (from earlier)
ax = axes[0]
ax.scatter(z_test_ae[:, 0], z_test_ae[:, 1], c=y_test, cmap="tab10", s=3, alpha=0.5)
ax.set_title("Autoencoder latent space", fontsize=13)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")

# RIGHT: VAE latent
ax = axes[1]
sc = ax.scatter(mu_test[:, 0], mu_test[:, 1], c=y_test, cmap="tab10", s=3, alpha=0.5)
ax.set_title("VAE latent space", fontsize=13)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")

fig.colorbar(sc, ax=axes, label="digit", ticks=range(10), shrink=0.8)
plt.suptitle("AE has gaps and irregular clusters; VAE is smooth and centred",
             fontsize=12, y=1.02)
plt.tight_layout(); plt.show()


The VAE latent space is:
- **Centred** around the origin (thanks to the KL term)
- **Smooth** — no big empty gaps
- **Organised** — digit classes form distinct but overlapping regions

All of this happened **without labels** — the model never saw digit labels during training!


### 10.2 Decode a grid — the generative manifold

In [ ]:
@nnx.jit
def decode(model, z):
    return jax.nn.sigmoid(model.decoder(z))

n_grid = 15
z1 = jnp.linspace(-3, 3, n_grid)
z2 = jnp.linspace(-3, 3, n_grid)
zz1, zz2 = jnp.meshgrid(z1, z2)
zs = jnp.stack([zz1, zz2], axis=-1).reshape(-1, 2)
imgs = np.asarray(decode(vae, zs)).reshape(n_grid, n_grid, 28, 28)

canvas = np.zeros((n_grid * 28, n_grid * 28))
for i in range(n_grid):
    for j in range(n_grid):
        canvas[(n_grid - 1 - i) * 28:(n_grid - i) * 28,
               j * 28:(j + 1) * 28] = imgs[i, j]

plt.figure(figsize=(7, 7))
plt.imshow(canvas, cmap="gray_r", extent=[-3, 3, -3, 3])
plt.xlabel("$z_1$", fontsize=12); plt.ylabel("$z_2$", fontsize=12)
plt.title("VAE decoded grid — every point generates a digit!", fontsize=13)
plt.tight_layout(); plt.show()


Every point in the 2D latent space maps to a recognisable digit. As you move smoothly through the space, digits **morph** continuously — this is the power of the VAE.


## 11. Latent space interpolation

One of the coolest things about VAEs: we can **smoothly morph** between any two digits by interpolating in latent space.


In [ ]:
def interpolate(model, x1, x2, n_steps=10):
    mu1, _ = model.encoder(x1[None])
    mu2, _ = model.encoder(x2[None])
    alphas = jnp.linspace(0, 1, n_steps)
    z_interp = jnp.array([(1 - a) * mu1[0] + a * mu2[0] for a in alphas])
    imgs = jax.nn.sigmoid(model.decoder(z_interp))
    return imgs

# Pick pairs of digits to interpolate
pairs = [(0, 1), (3, 8), (4, 9), (2, 7)]

fig, axes = plt.subplots(len(pairs), 12, figsize=(14, len(pairs) * 1.5 + 0.5))
n_steps = 10

for row, (d1, d2) in enumerate(pairs):
    idx1 = np.where(y_test == d1)[0][0]
    idx2 = np.where(y_test == d2)[0][0]

    # Show original
    axes[row, 0].imshow(x_test[idx1].reshape(28, 28), cmap="gray_r")
    axes[row, 0].set_ylabel(f"{d1} → {d2}", fontsize=11)

    # Interpolated images
    imgs = np.asarray(interpolate(vae, jnp.asarray(x_test[idx1]),
                                       jnp.asarray(x_test[idx2]), n_steps))
    for j in range(n_steps):
        axes[row, j+1].imshow(imgs[j].reshape(28, 28), cmap="gray_r")

    # Show original
    axes[row, 11].imshow(x_test[idx2].reshape(28, 28), cmap="gray_r")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Smooth interpolation between digits in latent space", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


## 12. Generate brand-new digits

The whole point: sample random noise $z \sim \mathcal{N}(0, 1)$ and decode it into a new image.


In [ ]:
# Generate random digits from the prior
key = jr.PRNGKey(123)
z_random = jr.normal(key, (32, 2))
generated = np.asarray(decode(vae, z_random))

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].reshape(28, 28), cmap="gray_r")
    ax.axis("off")
plt.suptitle("32 brand-new digits generated from random noise!", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()


These digits **don't exist in the training data** — they were generated entirely from random Gaussian noise, passed through the learned decoder. This is the core capability of a generative model.


## 13. Experiment: the β-VAE trade-off

What happens if we change the **balance** between reconstruction and KL?

$$\text{Loss} = \text{Reconstruction} + \beta \times \text{KL}$$

- $\beta$ **small** (e.g. 0.1): mostly reconstruction → sharp images, but messy latent space
- $\beta = 1$: standard VAE — balanced
- $\beta$ **large** (e.g. 4.0): mostly KL → very smooth latent space, but blurry images


In [ ]:
results = {}
for beta in [0.1, 1.0, 4.0]:
    print(f"\n--- Training with beta = {beta} ---")
    model = VAE(latent=2, rngs=nnx.Rngs(int(100 * beta)))
    h = train_vae(model, x_train, n_epochs=10, batch_size=128, lr=1e-3,
                  beta=beta, seed=int(100*beta))
    results[beta] = model


In [ ]:
# Compare latent spaces and generated samples across betas
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, beta in enumerate([0.1, 1.0, 4.0]):
    model = results[beta]

    # Top row: latent space
    ax = axes[0, col]
    mu_t = np.asarray(encode_mean(model, jnp.asarray(x_test)))
    sc = ax.scatter(mu_t[:, 0], mu_t[:, 1], c=y_test, cmap="tab10", s=3, alpha=0.5)
    ax.set_title(f"$\\beta = {beta}$", fontsize=14, fontweight="bold")
    ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")
    ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)

    # Bottom row: decoded grid
    ax = axes[1, col]
    n_g = 10
    zg = jnp.stack(jnp.meshgrid(jnp.linspace(-3, 3, n_g),
                                jnp.linspace(-3, 3, n_g)), axis=-1).reshape(-1, 2)
    ig = np.asarray(decode(model, zg)).reshape(n_g, n_g, 28, 28)
    cv = np.zeros((n_g * 28, n_g * 28))
    for i in range(n_g):
        for j in range(n_g):
            cv[(n_g-1-i)*28:(n_g-i)*28, j*28:(j+1)*28] = ig[i, j]
    ax.imshow(cv, cmap="gray_r")
    ax.axis("off")

axes[0, 0].set_ylabel("Latent space", fontsize=12)
axes[1, 0].set_ylabel("Decoded grid", fontsize=12)

fig.colorbar(sc, ax=axes[0, :].tolist(), label="digit", shrink=0.7)
plt.suptitle("Effect of β: reconstruction quality vs latent regularity", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()


**What do you see?**

| β | Latent space | Generated images |
|---|---|---|
| **0.1** | Spread out, clusters separated far | Sharp but some gaps/blobs |
| **1.0** | Nice, centred, smooth | Good balance of clarity and coverage |
| **4.0** | Very tight around origin | Blurry — too much regularisation |

**A common misconception — let's be precise about $\beta$.**

It is tempting to say "$\beta = 1$ is the *correct* value and everything else is
wrong." That is **not** right. Here is the accurate statement:

- $\beta = 1$ recovers the **original ELBO** derived in §7.0, i.e. it is the exact
  variational lower bound on $\log p_\theta(x)$. So $\beta = 1$ is *special*, but
  only in the sense that it equals the plain ELBO.
- $\beta > 1$ (the **β-VAE** of Higgins et al., 2017) is a deliberate, legitimate
  choice: you give up some ELBO tightness in exchange for a more strongly
  regularised, often more **disentangled** latent space. The objective is then a
  lower bound that is looser but emphasises the prior more.
- $\beta < 1$ leans toward sharper reconstructions at the cost of latent structure.

In free-energy language, $\beta$ is just the **temperature** weighting the entropic
(KL) term: $F = \langle E \rangle + \beta\,\text{KL}$. None of these are "wrong" —
they are different points on a principled trade-off curve, and the right $\beta$
depends on what you want from the model.


## 14. Summary: the full VAE pipeline


In [ ]:
# Full VAE pipeline diagram
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 16); ax.set_ylim(0, 6); ax.axis("off")

# Training flow (top)
ax.text(8, 5.5, "Training: learn to reconstruct", fontsize=14, ha="center",
        fontweight="bold", color="#1565c0")

box(ax, 0.2, 3.0, 1.6, 1.5, "Input\n$x$", "#e3f2fd")
box(ax, 2.8, 3.0, 2.2, 1.5, "Encoder\n$\\phi$", "#bbdefb")
box(ax, 6.0, 3.8, 1.5, 0.7, "$\\mu$", "#e1bee7", fontsize=10)
box(ax, 6.0, 3.0, 1.5, 0.7, "$\\log\\sigma$", "#e1bee7", fontsize=10)
box(ax, 8.2, 3.0, 2.3, 1.5, "Reparam\n$z = \\mu + \\sigma\\epsilon$", "#fce4ec",
    fontsize=9)
box(ax, 11.2, 3.0, 2.2, 1.5, "Decoder\n$\\theta$", "#c8e6c9")
box(ax, 14.0, 3.0, 1.6, 1.5, "Output\n$\\hat{x}$", "#a5d6a7")

arrow(ax, 1.8, 3.75, 2.8, 3.75)
arrow(ax, 5.0, 4.1, 6.0, 4.1)
arrow(ax, 5.0, 3.4, 6.0, 3.4)
arrow(ax, 7.5, 3.75, 8.2, 3.75)
arrow(ax, 10.5, 3.75, 11.2, 3.75)
arrow(ax, 13.4, 3.75, 14.0, 3.75)

# Loss arrows
ax.annotate("Reconstruction\nloss (BCE)",
            xy=(14.8, 3.0), xytext=(14.8, 1.8),
            fontsize=10, color="#c62828", ha="center",
            arrowprops=dict(arrowstyle="-|>", color="#c62828", lw=1.5))
ax.annotate("KL loss",
            xy=(9.3, 3.0), xytext=(9.3, 1.8),
            fontsize=10, color="#1565c0", ha="center",
            arrowprops=dict(arrowstyle="-|>", color="#1565c0", lw=1.5))

# Generation flow (bottom)
ax.text(8, 1.2, "Generation: sample new data", fontsize=14, ha="center",
        fontweight="bold", color="#2e7d32")

box(ax, 3.5, 0.0, 2.2, 0.9, "$z \\sim \\mathcal{N}(0, I)$", "#e0f2f1")
box(ax, 7.5, 0.0, 2.2, 0.9, "Decoder $\\theta$", "#c8e6c9")
box(ax, 11.5, 0.0, 2.2, 0.9, "New image!", "#a5d6a7")

arrow(ax, 5.7, 0.45, 7.5, 0.45, color="#2e7d32")
arrow(ax, 9.7, 0.45, 11.5, 0.45, color="#2e7d32")

plt.tight_layout(); plt.show()


## 15. Physics demo: an Ising-VAE discovers the order parameter

So far the VAE was a nice generative toy on MNIST. Now we do something a physicist
actually cares about: **let the VAE find an order parameter on its own.**

We reuse the **2D Ising Metropolis Monte-Carlo sampler from Lectures 01 / 06** —
no new external data, just the synthetic configurations the sampler produces. The
Ising Hamiltonian (with $J=1$, $k_B=1$, dimensionless) is

$$H = -J \sum_{\langle i,j\rangle} s_i s_j, \qquad s_i \in \{-1, +1\},$$

and its order parameter is the magnetisation per spin
$m = \tfrac{1}{N}\sum_i s_i$, with $|m| \to 1$ for $T \ll T_c$ and $|m| \to 0$ for
$T \gg T_c$ (Onsager: $T_c = 2/\ln(1+\sqrt2)\approx 2.269$).

**The experiment.** Generate configurations at four temperatures bracketing $T_c$
— two ordered ($T=1.5, 2.0$), one near-critical/disordered ($T=2.5$), one hot
($T=3.5$) — train a 2D-latent VAE on them *without ever telling it the temperature
or the magnetisation*, and then check whether a **latent axis lines up with $|m|$**.
If it does, the VAE has rediscovered the Ising order parameter from raw spins. This
turns the "Discovering order parameters" row of the applications table into a
runnable result.


### 15.1 The 2D Ising Metropolis sampler (reused from L01/L06)

This is the same JAX Metropolis sweep we wrote earlier in the course: one sweep =
$L^2$ single-spin-flip attempts, accept with probability $\min(1, e^{-\beta\Delta E})$,
periodic boundary conditions, all JIT-compiled with `lax.fori_loop`.


In [ ]:
def init_lattice(key, L):
    # L x L lattice of random +/-1 spins.
    return 2 * jr.bernoulli(key, shape=(L, L)).astype(jnp.int8) - 1


def magnetization(spins):
    # Magnetisation per spin m = (1/N) sum_i s_i (signed).
    return jnp.mean(spins.astype(jnp.float32))


@jax.jit
def metropolis_sweep(spins, key, beta):
    # One Metropolis sweep = L^2 single-spin-flip attempts (periodic BC).
    L = spins.shape[0]
    N = L * L

    def step(i, carry):
        spins, key = carry
        key, k1, k2, k3 = jr.split(key, 4)
        x = jr.randint(k1, (), 0, L)
        y = jr.randint(k2, (), 0, L)
        s = spins[x, y]
        nn_sum = (spins[(x + 1) % L, y] + spins[(x - 1) % L, y]
                  + spins[x, (y + 1) % L] + spins[x, (y - 1) % L])
        dE = (2.0 * s * nn_sum).astype(jnp.float32)
        accept = (dE <= 0) | (jr.uniform(k3) < jnp.exp(-beta * dE))
        new_s = jnp.where(accept, -s, s)
        spins = spins.at[x, y].set(new_s)
        return (spins, key)

    spins, key = lax.fori_loop(0, N, step, (spins, key))
    return spins, key


# Quick smoke test
_s = init_lattice(jr.PRNGKey(0), 32)
_s, _ = metropolis_sweep(_s, jr.PRNGKey(1), beta=1.0)
print(f"sweep OK, lattice {_s.shape}, m = {float(magnetization(_s)): .3f}")


### 15.2 Generate a labelled dataset of Ising configurations

For each temperature we warm up, then record one configuration every few sweeps
to reduce autocorrelation. We keep the true $T$ and the magnetisation **only for
plotting and the checkpoint** — the VAE never sees them. We store both the
**signed** $m = \tfrac1N\sum_i s_i$ and its magnitude $|m|$, because the Ising
model has a $\mathbb{Z}_2$ symmetry ($s_i \to -s_i$ leaves $H$ unchanged): below
$T_c$ the system settles into either the all-up or the all-down branch, so $m$
takes values near $+1$ *and* $-1$. Total $\approx 4000$ configs of $32\times 32$
spins; runs in a few CPU-seconds, no GPU needed.


In [ ]:
def sample_ising_dataset(temps, n_per_T=1000, L=32, n_warmup=200,
                          thin=2, seed=0):
    # Generate Ising configs at several temperatures.
    # Returns:
    #   configs : (n_total, L*L) float32 in {0,1}  (spins -1->0, +1->1)
    #   temp_lbl: (n_total,) float32               true temperature (plots only)
    #   m_signed: (n_total,) float32               signed m  = (1/N) sum s_i
    #   m_abs   : (n_total,) float32               |m|       (checkpoint label)
    key = jr.PRNGKey(seed)
    all_cfg, all_T, all_ms = [], [], []
    for T in temps:
        beta = 1.0 / T
        key, ik = jr.split(key)
        spins = init_lattice(ik, L)
        for _ in range(n_warmup):
            spins, key = metropolis_sweep(spins, key, beta)
        for _ in range(n_per_T):
            for _ in range(thin):
                spins, key = metropolis_sweep(spins, key, beta)
            all_cfg.append(np.asarray(spins).reshape(-1))
            all_T.append(T)
            all_ms.append(float(magnetization(spins)))   # signed
    configs = np.stack(all_cfg).astype("float32")
    configs = (configs > 0).astype("float32")        # spins {-1,+1} -> bits {0,1}
    m_signed = np.array(all_ms, "float32")
    return configs, np.array(all_T, "float32"), m_signed, np.abs(m_signed)


# Two ordered (T<Tc), one near/above Tc, one hot.  Tc ~ 2.269
ising_temps = [1.5, 2.0, 2.5, 3.5]
ising_x, ising_T, ising_m_signed, ising_m = sample_ising_dataset(
    ising_temps, n_per_T=1000, L=32, n_warmup=200, thin=2, seed=0)

print(f"configs shape : {ising_x.shape}   (n_total, L*L)")
print(f"temp labels   : {ising_T.shape}, unique = {sorted(set(ising_T.tolist()))}")
print(f"signed m      : range = [{ising_m_signed.min():.3f}, {ising_m_signed.max():.3f}]")
print(f"|m| labels    : range = [{ising_m.min():.3f}, {ising_m.max():.3f}]")


In [ ]:
# Sanity check the physics:
#   (left 4) sample snapshots;  (right) <|m|> falls and m spreads to +/-1 below Tc.
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, T in zip(axes[:4], ising_temps):
    idx = np.where(ising_T == T)[0][0]
    ax.imshow(ising_x[idx].reshape(32, 32), cmap="gray_r")
    ax.set_title(f"$T={T}$\n$m={ising_m_signed[idx]:+.2f}$", fontsize=11)
    ax.axis("off")

ax = axes[4]
mean_m = [ising_m[ising_T == T].mean() for T in ising_temps]
ax.plot(ising_temps, mean_m, "o-", color="royalblue", label=r"$\langle |m| \rangle$")
ax.axvline(2.269, color="red", ls="--", alpha=0.7, label="$T_c$")
ax.set_xlabel("$T$ [$J/k_B$]"); ax.set_ylabel(r"$\langle |m| \rangle$")
ax.set_title("order parameter vs $T$", fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 15.3 Train a VAE on the Ising configurations

We reuse the **exact same `VAE` class** as for MNIST — the only change is the input
dimension ($32\times 32 = 1024$ instead of 784). The encoder/decoder, the
reparameterization trick, and `vae_loss` are unchanged. The model still never sees
$T$ or $m$: it only sees raw spin bitmaps.


In [ ]:
ising_dim = ising_x.shape[1]   # 1024
ising_vae = VAE(in_dim=ising_dim, hidden=256, latent=2, rngs=nnx.Rngs(0))

# `train_vae` is dimension-agnostic, so we can reuse it directly.
ising_hist = train_vae(ising_vae, ising_x, n_epochs=20, batch_size=128,
                       lr=1e-3, beta=1.0, seed=0)


### 15.4 Does a latent axis track the magnetisation?

Encode every configuration to its posterior mean $\mu = (z_1, z_2)$ and colour each
point by the true **signed** magnetisation $m$. Because of the $\mathbb{Z}_2$
symmetry, the natural order parameter the VAE recovers is $m$ itself: ordered
configs ($m\approx +1$ and $m\approx -1$) sit at opposite ends of a latent axis,
and disordered configs ($m\approx 0$) cluster in between. We also colour by
temperature, which is a coarse proxy for $|m|$.


In [ ]:
@nnx.jit
def encode_mean_ising(model, x):
    mu, _ = model.encoder(x)
    return mu

mu_ising = np.asarray(encode_mean_ising(ising_vae, jnp.asarray(ising_x)))
print(f"latent means shape: {mu_ising.shape}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Colour by signed m (the order parameter) ...
ax = axes[0]
sc = ax.scatter(mu_ising[:, 0], mu_ising[:, 1], c=ising_m_signed, cmap="coolwarm",
                s=6, alpha=0.7, vmin=-1, vmax=1)
fig.colorbar(sc, ax=ax, label="signed $m$")
ax.set_title("Latent space coloured by magnetisation $m$", fontsize=12)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")

# ... and by temperature (a proxy for |m|).
ax = axes[1]
sc2 = ax.scatter(mu_ising[:, 0], mu_ising[:, 1], c=ising_T, cmap="viridis",
                 s=6, alpha=0.7)
fig.colorbar(sc2, ax=ax, label="temperature $T$")
ax.set_title("Latent space coloured by temperature $T$", fontsize=12)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")

plt.suptitle("An unsupervised VAE arranges Ising configs by their order parameter",
             fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


### 15.5 Checkpoint exercise (TODO-3): quantify the discovery

A colourful scatter is suggestive but not a proof. Make it quantitative with the
**Pearson correlation** between each latent coordinate and the order parameter,
then take the better-aligned axis. The cleanest signal is against the **signed**
$m$ (which respects the $\mathbb{Z}_2$ branches); we also report $|m|$ for context.

> **TODO-3.** Fill in the Pearson-correlation computation between two 1D arrays.
> *Hint:* `np.corrcoef(a, b)[0, 1]`.
>
> **Physics checkpoint:** the demo **passes** if
> $\max\big(|r(z_1, m)|,\,|r(z_2, m)|\big) > 0.85$, i.e. at least one latent axis
> tracks the magnetisation almost linearly. If you fall short, train a few more
> epochs or re-sample more configs — the order parameter is the dominant mode of
> variation, so a converged VAE should expose it.


In [ ]:
# ---- TODO-3: correlation between latent axes and the order parameter ----
def pearson(a, b):
    # TODO-3: return the Pearson correlation coefficient between 1D arrays a, b.
    # r = ...                                  # <-- fill me in
    r = np.corrcoef(a, b)[0, 1]
    return float(r)

# Correlate each latent axis with the signed order parameter m.
r1 = pearson(mu_ising[:, 0], ising_m_signed)
r2 = pearson(mu_ising[:, 1], ising_m_signed)
best = max(abs(r1), abs(r2))
which = "z1" if abs(r1) >= abs(r2) else "z2"

print(f"corr(z1, m) = {r1:+.3f}    corr(z1, |m|) = {pearson(mu_ising[:, 0], ising_m):+.3f}")
print(f"corr(z2, m) = {r2:+.3f}    corr(z2, |m|) = {pearson(mu_ising[:, 1], ising_m):+.3f}")
print(f"best-aligned latent axis (vs signed m): {which}  (|r| = {best:.3f})")

passed = best > 0.85
print("\\nCHECKPOINT", "PASSED" if passed else "NOT PASSED",
      f"(need |r| > 0.85, got {best:.3f})")
assert passed, ("TODO-3 checkpoint: no latent axis tracks the magnetisation "
                "strongly enough. Train longer or sample more configurations.")


In [ ]:
# Confirm it visually: the magnetisation-aligned axis vs signed m.
z_best = mu_ising[:, 0] if abs(r1) >= abs(r2) else mu_ising[:, 1]

plt.figure(figsize=(6, 5))
sc = plt.scatter(z_best, ising_m_signed, c=ising_T, cmap="viridis", s=6, alpha=0.7)
plt.colorbar(sc, label="temperature $T$")
plt.xlabel(f"latent axis ${which}$ (VAE, unsupervised)")
plt.ylabel("true signed $m$ (physics)")
plt.title(f"The VAE's latent axis is a learned order parameter\\n"
          f"Pearson $|r| = {best:.3f}$", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Takeaway.** The VAE was never told about temperature, the critical point, or
the magnetisation. Trained only to compress and reconstruct raw spin bitmaps, it
allocated one latent coordinate to the single most important collective variable —
the **order parameter** $m$. The $\mathbb{Z}_2$ symmetry of the Ising model shows
up directly: the two ordered branches $m\approx\pm1$ land at opposite ends of that
axis. This is exactly the physicist's analogy from the course intro:
*latent space ↔ collective coordinates / order parameters*. The same trick
generalises — VAEs (and their cousins) are now used to hunt for order parameters
and reaction coordinates in systems where we **don't** already know the answer.


## 16. Where are VAEs used?

| Application | What the VAE does |
|---|---|
| **Crystal structure generation** (CDVAE) | Latent space encodes crystal symmetries; decode to propose new materials |
| **Molecular design** | Encode molecules to latent space, optimise properties, decode new molecules |
| **Discovering order parameters** | Train on Ising configs → a latent axis aligns with magnetisation — **we just did this in §15!** |
| **Anomaly detection** | High reconstruction error = unusual/unseen data |
| **Data compression** | Latent code is a compact representation |
| **Image editing** | Modify latent code → change specific features |

## 17. The generative model zoo — preview

VAE is one of several generative model families. Each has different trade-offs:

| Family | Key idea | We'll cover |
|---|---|---|
| **VAE** | Encode to distribution, sample, decode | **Today** |
| **Normalizing flow** | Invertible transformations, exact likelihood | Next lecture |
| **Diffusion model** | Iterative denoising from noise to data | Lecture 16 |
| **GAN** | Generator vs discriminator adversarial game | Lecture 17 |

## References

1. Kingma & Welling, *Auto-Encoding Variational Bayes*, ICLR 2014.
2. Doersch, *Tutorial on Variational Autoencoders*, arXiv:1606.05908.
3. Higgins *et al.*, *β-VAE: Learning Basic Visual Concepts with a Constrained
   Variational Framework*, ICLR 2017.
4. Wetzel, *Unsupervised learning of phase transitions: From principal component
   analysis to variational autoencoders*, Phys. Rev. E **96**, 022140 (2017).
5. Ising sampler reused from PHYG004 Lectures 01 & 06 (2D Ising Metropolis MC).

---

*End of Lecture 14.*
